In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

In [2]:
import sys
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import stepmodel, connectionscanmodel
from quetzal.io import excel
from quetzal.io import road as road_io
import os
import numpy as np
import pandas as pd
import geopandas as gpd
if MANUAL:
    %matplotlib inline

io_engine= 'pyogrio'

In [ ]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

Index(['description', 'unit', 'category', 'parameter', 'base',
       'base_timetables'],
      dtype='object')


# Create model

In [5]:
UTM_EPSG = 2154  # France
if var['run_mode']['mode'] == 'frequency':
    model = stepmodel.StepModel(epsg=UTM_EPSG, coordinates_unit='meter')
elif var['run_mode']['mode'] == 'timetables':
    model = connectionscanmodel.ConnectionScanModel(epsg=UTM_EPSG, coordinates_unit='meter')

In [6]:
def read_file(path,engine):
    gdf = gpd.read_file(path, engine=engine)
    if 'index' in gdf.columns:
        gdf = gdf.set_index('index')
    else:
        gdf.index.name = 'index'
    return gdf

def read_links(filename, engine):
    links = read_file(filename, engine=engine)
    with open(filename, 'r') as j:
        json_data = json.loads(j.read())
    json_prop = json_data['features'][0]['properties'].keys()
    missing_columns = set(json_prop).difference(links.columns)
    for col in missing_columns:
        if col != 'index':
            d = {f['properties']['index']: f['properties'].get(col) for f in json_data['features']}
            links[col] = links.index.map(d) 

    return links

In [7]:
model.links = read_links(ipath + f'pt/links.geojson',  engine=io_engine).to_crs(epsg=2154)
model.nodes = read_file(ipath + f'pt/nodes.geojson', engine=io_engine).to_crs(epsg=2154)

model.road_nodes = read_file(ipath + 'road/simple_rn.geojson', engine=io_engine).to_crs(epsg=2154)
model.road_links = read_file(ipath + 'road/simple_rl.geojson', engine=io_engine).to_crs(epsg=2154)

# HEADWAY, MODE, SPEED

In [8]:
model.links.drop_duplicates(subset=['a', 'b', 'trip_id', 'link_sequence'], keep='first', inplace=True)

In [9]:
pd.set_option('display.max_columns', None)

In [10]:
model.links['speed'] = model.links['length'] / model.links['time']
model.links = model.links[~model.links['route_type'].isin(['schoolbus', 'ferry'])]
model.nodes = model.nodes.loc[model.nodes.index.isin(model.links['a'].unique().tolist() + model.links['b'].unique().tolist())]
model.integrity_fix_sequences()

dropped broken sequences: ['78_TBM_3_0']


In [11]:
# TODO Louison: vérifier les séquences droppées
### -> résolu

In [12]:
### model.road_links["capacity"] = model.road_links["lanes"] * 1000  
# TODO Louison: mettre la capacité des liens routiers
### -> résolu
# TODO Louison : supprimer le réseau tertiaire                
### à faire dans le réseau de simplification du réseau routier

# EXPORT

In [13]:
model.to_zippedpickles(mpath + r'{}/all_networks'.format(SCENARIO))

broken_sequences: 100%|██████████| 11/11 [00:01<00:00, 10.62it/s]


In [14]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

In [ ]:
sm.integrity_test_all(errors='ignore')

passed: integrity_test_circular_lines
not performed: integrity_test_collision


8106it [00:01, 5331.47it/s]


passed: integrity_test_dead_ends
not performed: integrity_test_duplicate_volumes
passed: integrity_test_isolated_roads
passed: integrity_test_nodeset_consistency


8106it [00:03, 2505.38it/s]


passed: integrity_test_road_network
passed: integrity_test_road_nodeset_consistency
passed: integrity_test_sequences


In [ ]:
brt_links = sm.links.loc[sm.links['route_type']=='brt']
brt_links.groupby('route_id')['length'].sum() / brt_links.groupby('route_id')['time'].sum() * 3.6

Series([], dtype: float64)